# 4.1 Functions User-defined

**Prerequisites:** 03 Flow Control Statement (all notebooks)
**Target:** Python 3.12+ (notes flag 3.13/3.14 differences)

### What you'll learn
- Defining and calling functions; docstrings
- Parameters: required, default, `*args`, `**kwargs`, packing and unpacking
- **Positional-only (`/`) and keyword-only (`*`) parameters**
- **Type hints on signatures** — and why Python does not enforce them
- Returning one value, several values, or nothing
- **Scope and the LEGB rule**, with `global` and `nonlocal`
- How arguments are passed — call by object reference
- Worked, real-world examples: predicates, gcd, retry backoff, a password checker
- Recursion, its depth limit, and memoisation with `@cache`
- `functools.partial`

---

## Functions

A **function** is a named, reusable block of code that runs only when it is called.

**Analogy:** a function is like a saved routine on a coffee machine. You program the steps once ("espresso: grind, tamp, 25-second shot"), then press one button whenever you want the result — same steps, any number of times, without re-describing them.

### Why functions exist
- **Reuse** — write the logic once, call it everywhere (in the same program, or from other programs via modules — see **07**).
- **Naming** — `retry_connection()` tells a reader *what* happens without making them read *how*.
- **Isolation** — each function gets its own local variables, so its internals cannot trample the rest of the program (that is *scope*, later in this notebook).
- **Testing** — a small function with inputs and a return value is exactly the unit that unit tests test.

Python ships **builtin** functions (`print()`, `len()`, `max()` — the subject of **4.2**); everything else we write ourselves: **user-defined functions**.

### Anatomy of a function

```python
def func_name(parameters):        # header: `def` + name + parameter list
    """Docstring."""              # optional — but write one
    block_of_code                 # body: indented statements
    return value                  # optional: hand a result back

func_name(arguments)              # a call runs the body
```

- Don't shadow a builtin name — `def len(...)` hides the real `len` for the rest of the module.
- Parameters are optional; the parentheses are not.
- The body is delimited by **indentation** — Python organises code with whitespace.

### Docstrings
- A **docstring** ("documentation string") is a string literal placed as the **first statement** of a module, function, class, or method.
- Convention: triple quotes; first line a one-sentence summary of what the function *does or returns*.
- It is not a comment — it is **stored on the function** as the `__doc__` attribute, which is what `help()`, IDE tooltips and documentation generators read.
- **When it matters:** any function whose behaviour isn't obvious from its name — which, on a team, is most of them.

## Defining and calling

### A function without parameters
The simplest case: no inputs, no return value — just a named action.

In [ ]:
def greeting():  
    '''Prints a Hello statement.'''
    print("Hello")

In [ ]:
print(greeting.__doc__)

In [ ]:
greeting()

In [ ]:
# A function is an object like any other — it has a type, a name, attributes...
print(type(greeting))
print(greeting.__name__)
greeting()   # ...and being callable is just one of the things you can do with it

- **Redefining a name:** `def` simply binds a function object to a name — so a second `def speech(): ...` rebinds the name and the first definition is gone.
    - Statically-typed languages (C, Java) reject the duplicate at compile time; Python happily replaces it.
    - ⚠️ In a notebook this bites for real: re-running an old cell silently restores an old definition.

In [ ]:
def speech():
    print("Welcome to this python class")
    
speech()

def speech():
    print("We will move forward towards discussion about python")
    
speech()

### Parameters
Parameters are what make a function reusable across different inputs. Three kinds, from strictest to loosest:

1. **Required** (non-default) — the caller must supply a value
2. **Default** — the caller may omit it; the default fills in
3. **Variable count** (`*args` / `**kwargs`) — the caller may pass any number

#### Required parameters
No `=` in the header. Omitting the argument at the call site is a `TypeError` — Python refuses to guess.

In [ ]:
def greeting(name):
    print('Hello '+name)
    
# greeting() #TypeError: 1 argument required
greeting('Aditya') # argument gets assigned to the local variable name once passed to the function.
greeting('Shiva')
# greeting('Preeti','Sweety') #TypeError: Only 1 argument required

#### Default values
A parameter with `= value` in the header becomes optional at the call site.

**Why/when:** defaults encode the *sensible common case* so routine calls stay short — think configuration knobs: `port=8080`, `timeout=5.0`, `retries=3`. Callers spell out only what differs from normal.

⚠️ Defaults are evaluated **once**, at `def` time — never use a mutable default like `items=[]` (see Pitfalls at the end, and **2.7**).

In [ ]:
def connect(host, port=8080):
    print(f'Connecting to {host}:{port} ...')

connect('db.internal')          # common case: the default port
connect('db.internal', 5432)    # special case: override it

#### Mixing required and default parameters
Parameters **with** defaults must come **after** parameters without — otherwise Python could not tell which positional argument belongs to which parameter.

```python
def connect(host, port=8080):     # OK: required first, defaults last
def connect(port=8080, host):     # SyntaxError: parameter without a default follows one with a default
```

In [ ]:
def connect(host, port=8080, timeout=5.0):
    print(f'{host}:{port} (timeout {timeout}s)')

connect('api.internal')             # both defaults apply
connect('api.internal', 443, 1.5)   # both overridden

### Passing arguments: positional vs keyword

Two ways to hand arguments over:

- **Positional** — matched to parameters left to right. Compact, but *position decides meaning*.
- **Keyword** — matched by `name=value`. Order-independent and self-documenting.

**When to prefer keywords:** any call where a reader cannot guess what a bare value means — compare `alert('payments', 'CRITICAL', 'sms')` with `alert(service='payments', severity='CRITICAL', channel='sms')`. (The `*` marker in the next section lets a function *require* keywords.)

In [ ]:
# Positional arguments — matched to parameters left to right
def alert(service, severity, channel='email'):
    print(f'[{severity}] {service} -> notify via {channel}')

alert('payments', 'CRITICAL')
alert('payments', 'CRITICAL', 'sms')
alert('CRITICAL', 'payments')   # ⚠️ runs fine, silently wrong — POSITION decides meaning, not the name

In [ ]:
# Keyword arguments — matched by name, in any order
def alert(service, severity, channel='email'):
    print(f'[{severity}] {service} -> notify via {channel}')

alert(service='payments', severity='CRITICAL')
alert(severity='WARN', channel='slack', service='auth')  # order no longer matters

---

### Positional-only and keyword-only parameters

> **Version note:** keyword-only parameters (`*`) arrived in **Python 3.0**; positional-only
> parameters (`/`) in **Python 3.8** (PEP 570).

By default a parameter can be passed either way. Sometimes you want to *forbid* one of them.

### Syntax breakdown

```
def f(a, b, /, c, d, *, e, f):
            |        |
            |        +-- everything AFTER * is keyword-only
            +----------- everything BEFORE / is positional-only
```

| Group | Passed as | Why you'd want it |
|---|---|---|
| Before `/` | Position only | The name is an implementation detail you may want to rename later. Most builtins work this way — `len(obj=[1])` is an error. |
| Between | Either | The default |
| After `*` | Keyword only | Forces self-documenting calls. `create_user("A", True, False)` is unreadable; `create_user("A", is_admin=True)` is not. |

**Real-world use case:** keyword-only is the standard treatment for **boolean flags and
options**. A bare `True` at a call site never explains itself.

In [ ]:
# ---- Keyword-only: everything after * MUST be named at the call site ----
def create_user(name, *, is_admin=False, send_email=True):
    return f"{name}: admin={is_admin}, email={send_email}"

print(create_user("Aditya", is_admin=True))
print(create_user("Priya", send_email=False))

try:
    create_user("Rahul", True)          # positional - refused
except TypeError as exc:
    print("\npositional flag:", exc)

print("\nWhy: create_user('Rahul', True, False) tells the reader nothing.")


# ---- Positional-only: everything before / CANNOT be named ----
def distance(x, y, /, unit="km"):
    return f"{(x ** 2 + y ** 2) ** 0.5:.2f} {unit}"

print("\n" + distance(3, 4))
print(distance(3, 4, unit="miles"))

try:
    distance(x=3, y=4)                  # named - refused
except TypeError as exc:
    print("named coords  :", exc)


# ---- All three kinds in one signature ----
def full(a, b, /, c, d, *, e, f):
    return dict(a=a, b=b, c=c, d=d, e=e, f=f)

print("\n", full(1, 2, 3, d=4, e=5, f=6))

print("""
  a, b   positional-only  - pass by position, never by name
  c, d   either           - position or keyword, your choice
  e, f   keyword-only     - must be named
""")

# Many builtins are positional-only, which is why this fails:
try:
    len(obj=[1, 2, 3])
except TypeError as exc:
    print("len(obj=...):", exc)

### Arbitrary numbers of arguments: packing

Sometimes you cannot know how many arguments a caller will send — `print()` itself accepts any number.

- `*args` **packs** the extra positional arguments into a **tuple**
- `**kwargs` packs the extra keyword arguments into a **dict**

**Why/when:** aggregation helpers that take "as many as you like", and — the big one — **forwarding**: wrappers and decorators (see **4.4**) accept `*args, **kwargs` and pass them straight through to the wrapped function without needing to know its signature.

In [ ]:
def add(*args):  # the positional arguments get packed into a tuple before being passed into the function.
    print('Tuple:',args)
    s = 0
    for i in args:
        s = s+i
    print('Sum:',s)        
    
add(3,4)
add(3,4,5)

In [ ]:
def add(**kwargs): # the keyword arguments get packed into a dictionary before being passed into the function.
    print('Dictionary:',kwargs)
    s = 0
    for i in kwargs.values():
        s = s+i
    print('Sum:',s)
    
add(x=2, y=5)
add(x=2, y=5, z=3)

In [ ]:
# All the kinds together: required, then *args, then **kwargs
def register_device(name, ip, *tags, **metadata):
    print('Required :', name, ip)
    print('Tags     :', tags)        # extra positionals -> tuple
    print('Metadata :', metadata)    # extra keywords    -> dict

register_device('edge-01', '10.0.0.7', 'router', 'lab', vendor='HFCL', rack=12)

#### Unpacking: the same stars, at the call site

At a **call**, `*` and `**` do the reverse: they explode a sequence / dict into individual arguments.

**Why/when:** arguments assembled as *data* — a settings dict loaded from a file, a tuple built by another function — can be applied to a call in one step: `connect(*conn_args, **conn_opts)`.

In [ ]:
def connect(host, port, user, timeout):
    print(f'{user}@{host}:{port} (timeout={timeout}s)')

conn_args = ['db.internal', 5432]                  # a sequence unpacks with *
conn_opts = {'user': 'svc_nms', 'timeout': 3.0}    # a dict unpacks with **
connect(*conn_args, **conn_opts)

### `return` — handing a result back

- `return expr` ends the function immediately and gives `expr` to the caller.
- A function with **no** `return` (or a bare `return`) returns `None`.
- Code after a `return` on the same path never runs.
- ⚠️ **`print` is not `return`.** `print` shows a value to a *human* and produces nothing; `return` hands the value to the *program*, where it can be stored, compared, or passed on. A function that only prints cannot be used in an expression — `total = add(2, 3)` would quietly be `None`.

In [ ]:
def add(n1, n2):
    s = n1 + n2
    print('Sum is', s)
    return s

# Interactive variant:
# x = int(input("Enter 1st no.: "))
# y = int(input("Enter 2nd no.: "))
x, y = 7, 5
res = add(x, y)
print(res, type(res))
if res % 2 == 0:
    print('Even')
else:
    print('Odd')

In [ ]:
# Implicit return statement
def add(n1, n2):
    s = n1 + n2
    print('Sum is', s)

# Interactive variant:
# x = int(input("Enter 1st no.: "))
# y = int(input("Enter 2nd no.: "))
x, y = 7, 5
res = add(x, y)
print(res, type(res))

### Returning several values at once

A function often computes *related* facts in one pass — a stats helper produces the minimum, maximum and average together. Python returns exactly one object, so several values travel **packed in a container**. Four options, in rising order of structure.

Running scenario for the next cells: summarising a batch of **response-time samples** (milliseconds).

#### 1) As a tuple — the idiomatic default

In [ ]:
def latency_summary(samples):
    return (min(samples), max(samples), sum(samples) / len(samples))  # one tuple, three facts

pings = [12, 7, 22, 9, 15]           # response times in ms
op = latency_summary(pings)
print(op, type(op))
print(op[0])   # fastest
print(op[1])   # slowest

In [ ]:
# Tuple unpacking at the call site — the reason tuples are the idiomatic choice
fastest, slowest, average = latency_summary(pings)
print(fastest)
print(slowest)
print(average)

#### 2) As a list
Same idea, but mutable — use it when the caller is expected to modify or extend the result. For fixed, record-shaped results, prefer a tuple or a dict.

In [ ]:
def latency_summary_list(samples):
    return [min(samples), max(samples), sum(samples) / len(samples)]

op = latency_summary_list(pings)
print(op, type(op))
print(op[0])
print(op[1])

#### 3) As a dictionary
The result labels itself — `op['avg']` reads better than `op[2]`, and adding a field later does not break existing callers the way it breaks positional unpacking.

In [ ]:
def latency_summary_dict(samples):
    return {'min': min(samples), 'max': max(samples), 'avg': sum(samples) / len(samples)}

op = latency_summary_dict(pings)
print(op, type(op))
print(op['avg'])

#### 4) As an object
For results that carry behaviour (or many fields), return an instance of a class — or, in modern code, a `typing.NamedTuple` / `@dataclass` (see **05 OOPs**), which give you names *and* tuple-style unpacking.

In [ ]:
class LatencyStats:
    def __init__(self, samples):
        self.fastest = min(samples)
        self.slowest = max(samples)
        self.average = sum(samples) / len(samples)

def latency_summary_obj(samples):
    return LatencyStats(samples)

op = latency_summary_obj(pings)
print(op.fastest)
print(op.slowest)
print(round(op.average, 1))

---

### Type hints on signatures

> **Version note:** annotations arrived in **Python 3.0**, and PEP 484 gave them meaning in
> **3.5**. They appear nowhere in the original notes because they were far less common in 2019.

A type hint records what a parameter is *supposed* to be, and what the function *gives back*.

### Syntax breakdown

```
def apply_discount(price: float, percent: float = 10.0) -> float:
                     |      |            |         |        |
                     |      |            |         |        +-- return type
                     |      |            |         +----------- default value
                     |      |            +--------------------- annotation
                     |      +---------------------------------- annotation
                     +------------------------------------------ parameter
```

### ⚠️ The single most important fact

**Python does not enforce type hints at run time.** `def f(x: int)` will accept a string,
a list, or anything else without complaint. Annotations are:

- **Documentation** that cannot drift out of date silently
- **Input to tools** — `mypy`, `pyright`, and your editor's autocomplete
- **Readable by the program itself**, via `__annotations__`

They are checked by *you running a checker*, not by the interpreter.

Common annotations for now (the full treatment is **4.5** and **16**):

| Hint | Means |
|---|---|
| `int`, `str`, `float`, `bool` | A single value of that type |
| `list[int]`, `dict[str, float]` | Containers (3.9+ — no `typing.List` needed) |
| `str \| None` | Either a `str` or `None` (3.10+) |
| `-> None` | Returns nothing useful |

In [ ]:
from typing import Optional

# A fully annotated function
def apply_discount(price: float, percent: float = 10.0) -> float:
    """Return `price` reduced by `percent`."""
    return price * (1 - percent / 100)

print(apply_discount(100))
print(apply_discount(100, 25))

# Annotations are stored, not enforced
print("\n__annotations__:", apply_discount.__annotations__)

# ⚠️ Python does NOT check them at run time
print("with a string  :", apply_discount.__annotations__["price"], "declared,",
      type("100").__name__, "accepted:")
try:
    print("  ", apply_discount("100"))
except TypeError as exc:
    print("   fails only when the ARITHMETIC fails:", exc)


# Container types (3.9+ builtin generics - no typing.List needed)
def average(marks: list[float]) -> float:
    return sum(marks) / len(marks)

def tally(words: list[str]) -> dict[str, int]:
    counts: dict[str, int] = {}
    for w in words:
        counts[w] = counts.get(w, 0) + 1
    return counts

print("\naverage:", average([88, 91, 79]))
print("tally  :", tally(["a", "b", "a"]))


# "Maybe a value, maybe None" - two spellings, the second is modern (3.10+)
def find_user(uid: int) -> Optional[str]:
    return {1: "Aditya"}.get(uid)

def find_user_modern(uid: int) -> str | None:
    return {1: "Aditya"}.get(uid)

print("\nfound  :", find_user(1))
print("missing:", find_user_modern(99))


# A function that returns nothing is annotated -> None
def log(message: str) -> None:
    print(f"  LOG: {message}")

log("annotations are documentation the tools can check")

### Scope of a Variable: the LEGB rule

The **scope** of a name is the region of the program where it can be reached. When Python
meets a name, it searches four namespaces **in order**, and stops at the first hit:

```
    L  ->  Local        names inside the current function
    E  ->  Enclosing    names in any enclosing function (closures)
    G  ->  Global       names at module level
    B  ->  Built-in     print, len, str, ... always last
```

**Analogy:** looking for a document. Check your desk (Local), then your team's shelf
(Enclosing), then the office archive (Global), then the public library (Built-in). You stop
as soon as you find one — which is exactly why a local `list = [...]` hides the builtin.

### Reading is easy; writing needs permission

Reading an outer name Just Works. **Assigning** to a name makes it local to that function
for the whole function body — unless you say otherwise:

| Keyword | Means |
|---|---|
| `global x` | Assignments to `x` go to the **module** namespace |
| `nonlocal x` | Assignments to `x` go to the nearest **enclosing function** |

- **Global variables** are defined at module level and readable anywhere.
- **Local variables** are defined inside a function and vanish when it returns.

In [ ]:
x = "global"

def outer():
    x = "enclosing"

    def inner():
        x = "local"
        print("  inner sees :", x)          # L

    inner()
    print("  outer sees :", x)              # E (unchanged by inner)

outer()
print("module sees:", x)                     # G
print("built-in   :", len, " <- B, found last")


# ⚠️ Assigning ANYWHERE in a function makes the name local for the WHOLE function
count = 0

def broken():
    # print(count)     # UnboundLocalError - `count` is local because of the line below
    count = 1
    return count

print("\nbroken()     :", broken(), "| module count still", count)


# `global` - write to the module namespace
def with_global():
    global count
    count += 1

with_global()
print("after global :", count)


# `nonlocal` - write to the nearest ENCLOSING function
def counter():
    total = 0

    def increment():
        nonlocal total          # without this: UnboundLocalError
        total += 1
        return total

    increment(); increment(); increment()
    return total

print("nonlocal     :", counter())

# Shadowing a builtin: legal, and a trap
def shadow():
    list = [1, 2, 3]            # the builtin is now hidden in this scope
    return list

print("\nshadow()     :", shadow())
print("builtin fine outside:", list("ab"))

In [ ]:
# Define local variable
def arithmetic(a,b):
    total=a+b #total,a,b are local variable
    print('Sum',total)
    
arithmetic(4,5)

In [ ]:
# print(total) #If we try to access the variable 'total' outside the function, we'll get NameError.

In [ ]:
# Define global variable
# Interactive variant: x = int(input("Enter no. "))
x = 10  # x is a global variable

def arithmetic(a, b):
    total = a + b + x
    print('Sum', total)

arithmetic(4, 5)

if x % 2 == 0:
    print(x, "is Even")
else:
    print(x, "is Odd")

In [ ]:
# Modify global variable inside function definition
# Interactive variant: x = int(input("Enter no. "))
x = 10

def arithmetic(a, b):
    x = x + 2  # ⚠️ 'x' is assigned in this function, so Python treats it as LOCAL
    total = a + b + x
    print('Sum', total)

try:
    arithmetic(4, 5)
except UnboundLocalError as e:
    print(f'UnboundLocalError: {e}')

# We can't rebind a global variable from inside a function like this: the
# assignment makes 'x' local for the WHOLE function body, so 'x + 2' tries to
# read a local that has no value yet. To really modify the global, declare it
# with the 'global' keyword — the next cell shows the fix.

In [ ]:
# Defining global variable inside function
# Interactive variant: x = int(input("Enter no. "))
x = 10

def arithmetic(a, b):
    global x   # the fix: now 'x' refers to the global variable
    x = x + 2
    total = a + b + x
    print('Sum', total)

arithmetic(4, 5)

if x % 2 == 0:
    print(x, "is Even")
else:
    print(x, "is Odd")

In [ ]:
# Defining global and local variable with same name
# Interactive variant: x = int(input("Enter no. "))
x = 10

def arithmetic(a, b):
    x = 3  # a brand-new LOCAL x; the global x is untouched
    total = a + b + x
    print('Sum', total)

arithmetic(4, 5)

if x % 2 == 0:
    print(x, "is Even")
else:
    print(x, "is Odd")

### How arguments are passed: *call by object reference*

> **Heading corrected.** The original called this "Pass by Reference". Python is **neither**
> pass-by-value nor pass-by-reference in the C or C++ sense, and calling it either one leads
> to wrong predictions. The usual name is **call by object reference** (or "call by
> assignment").

- In Python, every variable name is a **reference to an object**.
- Calling a function **binds the parameter names to the same objects** the caller passed.
  Nothing is copied.
- So the function and the caller share objects — but the function's *names* are its own.

The practical rule follows directly:

| What the function does | Caller sees it? |
|---|---|
| **Rebinds** the parameter (`x = 31`) | ❌ No — only the local name moved |
| **Mutates** the object (`y.append(32)`) | ✅ Yes — same object |

Which means the answer depends entirely on whether the argument is **mutable**:

| Argument type | Can the function change it? |
|---|---|
| `int`, `str`, `tuple`, `frozenset` (immutable) | No — there is no operation that could |
| `list`, `dict`, `set` (mutable) | Yes, if it calls a mutating method |

This is the same names-vs-objects model from **2.7 Mutability, Copying, Nesting &
Unpacking** — arguments are just another form of assignment.

### Mutable Vs Immutable Arguments:

In [ ]:
def func(x,y,z):
    x = 31
    y.append(32)
    z = [33] # new reference
    print(x,'\n',y,'\n',z)

In [ ]:
a = 11 # im-mutable object
b = [12] # mutable object
c = [13] # mutable object 
func(a,b,c)
print(a,'\n',b,'\n',c)

#### Defensive copies: protecting the caller

If a function must not change the list it was given (or you don't trust it not to), pass a **copy**: `func(b[:])` or `func(b.copy())`. The function then mutates its private copy and the caller's data survives.

⚠️ A slice copy is *shallow* — nested objects inside are still shared (see **2.7** for `deepcopy`).

In [ ]:
def func(x,y,z):
    x = 31
    y.append(32)
    z = [33] # new reference
    print(x,'\n',y,'\n',z)

In [ ]:
a = 11 # im-mutable object
b = [12] # mutable object
c = [13] # mutable object 
func(a,b[:],c) # i.e, create an explicit copy of mutable object 'b' in the function.
                     # Now 'y' in func() refers to a different object which was initially a copy of b
print(a,'\n',b,'\n',c)

### Deleting a function

`def` binds a name; `del` unbinds it. After `del display`, the name is gone from the namespace and calling it raises `NameError` — the same error as any undefined name.

**When would you ever?** Rarely — occasionally to clean a module's (or notebook's) namespace of helpers that shouldn't be importable. It is shown here because it completes the mental model: *functions are just objects bound to names*.

In [ ]:
def display():
    print("Hello")
    
display()

In [ ]:
del display
display()

---

## Putting it together — worked examples

Everything so far — parameters, returns, scope — now gets exercised on small, complete functions. Each example is framed the way you'd meet it in working code; the classics (factorial, fibonacci) stay because they are the standard way to learn the shapes, and each is bridged to a real use.

### Predicate functions: `is_even`

A **predicate** is a function that answers a yes/no question with a `bool`. They are everywhere in real code — `is_valid(config)`, `is_expired(token)`, `has_capacity(queue)` — and they slot straight into `if`, `filter()` and `any()`/`all()` (see **4.2**).

Note that it **returns** the bool rather than printing a sentence — that is what makes it composable.

In [ ]:
def is_even(n):
    '''True if n is divisible by 2.'''
    return n % 2 == 0

print(is_even(17), is_even(42))

# A real use: parity is the simplest way to split work across two workers.
for task_id in range(101, 106):
    worker = 'worker-A' if is_even(task_id) else 'worker-B'
    print(f'task {task_id} -> {worker}')

### Prime numbers in a range

The classic drill — with a real anchor: primes matter in software because hash tables like prime bucket counts (fewer collision patterns), and public-key cryptography is built on large primes.

Structure note: a **predicate** `is_prime(n)` does one job; `primes_between` reuses it. Two small functions beat one big nested loop — each is testable on its own.

In [ ]:
def is_prime(n):
    '''True if n has exactly two divisors: 1 and itself.'''
    if n < 2:
        return False
    for d in range(2, int(n ** 0.5) + 1):   # trial division up to sqrt(n) is enough
        if n % d == 0:
            return False
    return True

def primes_between(lower, upper):
    '''Primes strictly between lower and upper.'''
    return [n for n in range(lower + 1, upper) if is_prime(n)]

print(primes_between(10, 50))

# e.g. choosing a prime bucket count for a hash table of ~40 slots:
print('next prime >= 40 :', primes_between(39, 60)[0])

### Greatest common divisor (HCF / GCD)

The HCF (highest common factor, a.k.a. GCD) of two numbers is the largest integer dividing both. Real uses: **simplifying ratios** (a 1920×1080 screen is "16:9" because their gcd is 120) and **aligning periodic jobs** (two tasks on 54 s and 24 s cycles coincide every `lcm = a*b // gcd` seconds).

The loop below is the readable drill version; the standard library ships `math.gcd` (Euclid's algorithm — far faster).

In [ ]:
def compute_hcf(a, b):
    smaller = min(a, b)
    hcf = 1
    for i in range(1, smaller + 1):
        if a % i == 0 and b % i == 0:
            hcf = i
    return hcf

import math

print('HCF of 54 and 24 :', compute_hcf(54, 24))
print('math.gcd agrees  :', math.gcd(54, 24))

# Real use 1: simplify an aspect ratio
w, h = 1920, 1080
g = math.gcd(w, h)
print(f'{w}x{h} -> {w // g}:{h // g}')

# Real use 2: when do a 54s job and a 24s job fire together?
print('every', 54 * 24 // math.gcd(54, 24), 'seconds  (the LCM)')

### Factorial (iterative)

`n! = n*(n-1)*...*2*1` — the classic. Its real face: `n!` counts **orderings**. "How many distinct orders could these 6 migration steps run in?" is `6!` — which is why brute-forcing permutations gets hopeless fast.

In [ ]:
def fact(num):
    f = 1
    for i in range(num, 0, -1):
        f *= i
    return f

print('6!  =', fact(6), ' -> possible orderings of 6 migration steps')
print('10! =', fact(10), ' -> why brute-forcing orderings does not scale')

### Fibonacci series (iterative)

`0 1 1 2 3 5 8 ...` — each term is the sum of the previous two. Beyond the textbook, the growth pattern is genuinely used: **fibonacci backoff** spaces out retry attempts (1 s, 1 s, 2 s, 3 s, 5 s, 8 s, ...) — gentler than doubling, but still backing off.

This version **returns the list** instead of printing — the caller decides what to do with it.

In [ ]:
def fibo(num):
    '''First `num` fibonacci terms, starting 0, 1.'''
    terms = []
    a, b = 0, 1
    for _ in range(num):
        terms.append(a)
        a, b = b, a + b
    return terms

print(fibo(10))

# Real use: retry delays that grow, but not brutally
delays = [t for t in fibo(8) if t > 0]      # drop the leading 0
print('retry after (s):', delays)

### Character-class counting: a password-policy checker

The drill "count upper/lower case and spaces in a string" is, in the real world, the core of a **password or input policy check**. One pass, several counters — returned as a dict (see "returning several values" above), so the caller can apply any policy it likes.

In [ ]:
def char_classes(s):
    counts = {'upper': 0, 'lower': 0, 'digit': 0, 'other': 0}
    for ch in s:
        if ch.isupper():
            counts['upper'] += 1
        elif ch.islower():
            counts['lower'] += 1
        elif ch.isdigit():
            counts['digit'] += 1
        else:
            counts['other'] += 1
    return counts

print(char_classes('Refactor The Legacy Parser'))

c = char_classes('Adity@123')
ok = c['upper'] >= 1 and c['digit'] >= 1 and c['other'] >= 1
print('Adity@123 ->', c)
print('passes policy (needs upper + digit + symbol):', ok)

### String transformation: `spin_words`

A pure **string-pipeline** function: split → transform each word → rejoin. The same shape as real text jobs (normalising log fields, slugifying titles, masking tokens). The transform uses the builtin `reversed()` — a quick detour to see what it gives us.

In [ ]:
print(reversed.__doc__)
print(list(reversed('Wolf')))   # an iterator over the sequence, back to front

In [ ]:
def spin_words(txt):
    words = txt.split()
    new = []
    for w in words:
        if len(w) >= 5:  # words with 5 or more characters get reversed
            new.append(''.join(reversed(w)))
        else:
            new.append(w)
    return ' '.join(new)

# Interactive variant: msg = input("Enter your msg: ")
msg = "Hey fellow warriors"
print(spin_words(msg))

## Recursion

**Recursion** is a function calling itself, each call working on a *smaller piece* of the problem, until a **base case** stops the chain.

**Analogy:** to total the size of a folder, you add up each file — plus the size of each *subfolder*, which is the same problem one level down. Recursion is the natural shape wherever data nests: directory trees, JSON documents, org charts, comment threads.

Every correct recursive function has two parts:

| Part | Job | If it's missing |
|---|---|---|
| **Base case** | The input small enough to answer directly | Infinite recursion → `RecursionError` |
| **Recursive case** | Call yourself on a *strictly smaller* input | Same — the input must shrink toward the base |

⚠️ Anything recursion can do, a loop (plus an explicit stack) can also do — and in Python the loop is cheaper (see "Recursion limits" below). Choose recursion when it *mirrors the data's shape*, not for menus or counters.

### Factorial, recursively

`n! = n × (n-1)!` with `0! = 1` — the mathematical definition *is already recursive*, which is why factorial is the standard first example.

In [ ]:
def fact_r(num):
    if (num == 0 or num == 1):  # base case
        return 1
    return num * fact_r(num - 1)  # recursive case

In [ ]:
# Interactive variant: num = int(input("Enter no.: "))
num = 5
print(fact_r(num))

- The base case that defines to get out of the Python recursion is when n is equal to 0 or 1. 
    - In that case, the function will return 1.
- Otherwise, it will return n multiplied by factorial(n-1). This is a recursive call to itself. 
- So this is how it goes:
```python
factorial(5)
=5*factorial(4)
=5*4*factorial(3)
=5*4*3*factorial(2)
=5*4*3*2*factorial(1)
=5*4*3*2*1
=5*4*3*2
=5*4*6
=5*24
=120
```
- The calls pile up on the call stack, which is last-in-first-out (LIFO): the deepest call `factorial(1)` finishes first, and each pending multiplication then completes from the innermost call outward, as the trace above shows.

### Fibonacci, recursively

Elegant to write — and, as written, **exponentially slow**: `fibo_r(n)` recomputes the same sub-results over and over. The shape is still worth knowing; the fix (`@cache`) is two sections down.

In [ ]:
# Using recursion
def fibo_r(num):
    if (num == 1 or num == 2):
        return num - 1
    return (fibo_r(num - 2) + fibo_r(num - 1))

In [ ]:
# Interactive variant: n = int(input("Enter no. of terms: "))
n = 10
for i in range(1, n + 1):
    print(fibo_r(i), end=' ')

### Decimal → binary, recursively

Repeatedly halving a number *is* a shrinking subproblem, so base conversion recurses naturally: recurse on `num // 2` first, then print `num % 2` on the way back out (most-significant bit first). This is what `bin()` does — and binary output is what you read in **permission bits and bitmasks** (`chmod 755`-style flags).

In [ ]:
def dec_bin(num):
    if num > 1:
        dec_bin(num // 2)      # recurse first...
    print(num % 2, end='')     # ...print on the way back out

decimal = 34
dec_bin(decimal)
print('\nbin() agrees:', bin(decimal))

---

### Recursion limits, and memoisation with `@cache`

Two things the recursion section above needs.

**1. Python has a recursion limit.** Each call consumes a stack frame, and CPython caps the
depth (1000 by default) to turn a runaway recursion into a `RecursionError` rather than a
crash. Python also has **no tail-call optimisation**, so a recursive solution is never
cheaper than the equivalent loop.

**2. Naive recursive fibonacci is exponential.** `fib(30)` recomputes `fib(10)` thousands of
times. One decorator fixes it.

> **Version note:** `functools.cache` was added in **3.9**. It is `lru_cache(maxsize=None)`
> with a friendlier name.

The same decorator earns its keep well outside maths: memoising any **pure, repeatedly-called** function — a per-route price lookup, a parsed-config getter, an expensive API-cost calculation — turns repeat calls into dictionary hits.

In [ ]:
import sys, time
from functools import cache, lru_cache

# ---- Recursion has a hard limit ----
print("recursion limit:", sys.getrecursionlimit())

def countdown(n):
    if n == 0:
        return 0
    return countdown(n - 1)

try:
    countdown(100_000)
except RecursionError as exc:
    print("RecursionError:", exc)

print("\nPython has no tail-call optimisation - deep recursion ALWAYS costs stack frames.")
print("For deep problems, use a loop or an explicit stack instead.")


# ---- Memoising the naive fibonacci ----
def fib_naive(n):
    if n <= 2:
        return 1
    return fib_naive(n - 1) + fib_naive(n - 2)

@cache                      # 3.9+; @lru_cache(maxsize=None) on older versions
def fib_cached(n):
    if n <= 2:
        return 1
    return fib_cached(n - 1) + fib_cached(n - 2)

N = 30

start = time.perf_counter()
a = fib_naive(N)
naive_time = time.perf_counter() - start

start = time.perf_counter()
b = fib_cached(N)
cached_time = time.perf_counter() - start

assert a == b
print(f"\nfib({N}) = {a}")
print(f"  naive : {naive_time * 1000:8.2f} ms")
print(f"  cached: {cached_time * 1000:8.4f} ms")
print(f"  speedup: {naive_time / cached_time:,.0f}x")
print("\ncache stats:", fib_cached.cache_info())

# The cached version can go far beyond what the naive one could ever finish
print("\nfib(200) =", fib_cached(200))

# ⚠️ @cache only works for HASHABLE arguments, and only for PURE functions.
# Caching something that reads a file or the clock will return stale results.

## Partial functions — `functools.partial`

`partial(func, *fixed_args, **fixed_kwargs)` returns a **new callable with some arguments pre-filled**. Think of it as freezing the configuration and leaving only the interesting part open.

**Why/when:**
- Deriving **specialised helpers** from one general function (below: one polynomial family → one concrete curve)
- **Callbacks** that must take few arguments (schedulers, GUI buttons) but need context baked in
- Feeding a one-argument function to `map()` / `sorted(key=...)` when the real function takes several

In [ ]:
from functools import partial

def poly_quadratic(b0, b1, b2, x):  # intercept and coefficients of a degree-2 polynomial
    return (b0 + (b1 * x) + (b2 * (x ** 2)))

# Fix the coefficients; only x stays free. The result is ONE specific curve
# (a parabola) — evaluating it at different x values walks along the same curve.
y = partial(poly_quadratic, 2, 0.5, 0.1)
print(y(2))
print(y(5))

# A day-job example: one generic logger, three pre-configured severities
def log(level, message):
    print(f'[{level}] {message}')

info = partial(log, 'INFO')
warn = partial(log, 'WARN')
error = partial(log, 'ERROR')

info('service started')
warn('disk at 81%')
error('raid degraded')

---

## Capstone demos

Two longer, complete programs that combine parameters, returns, validation and state. Both are deterministic (no `input()`, seeded randomness), so they run end-to-end.

### 1) Lottery number matching
Draw 6 distinct digits, parse the player's guess, pay out by number of matches — sets, validation and a payout table in one function-driven flow.

In [ ]:
import random

# ⚠️ A set can't be initialised with {} — that literal makes a dict; use set().
# ⚠️ A plain `for i in range(6)` adding random digits may draw duplicates, and a
#    set silently drops them (no repetition allowed) — so loop UNTIL it holds
#    6 distinct digits.

def lottery_no(rng):
    """Draw 6 distinct digits from 0-9."""
    ticket_no = set()
    while len(ticket_no) < 6:
        ticket_no.add(rng.randint(0, 9))
    return ticket_no

def gamer_no(guess):
    """Parse '0,2,4,6,8,9' -> {0, 2, 4, 6, 8, 9}.

    Interactive variant:
    guess = input('Enter 6 non-repetitive comma separated digits from 0-9: ')
    """
    return {int(d) for d in guess.split(',')}

# Payout table: prize (Rs.) for the number of matched digits — zero pays zero.
PAYOUT = {0: 0, 1: 10, 2: 100, 3: 1_000, 4: 10_000, 5: 100_000, 6: 1_000_000}

def final_round(guess, seed=42):
    rng = random.Random(seed)  # seeded generator -> deterministic, repeatable demo
    ticket_no = lottery_no(rng)
    user_no = gamer_no(guess)
    # ⚠️ Sets are unordered — pairing two sets with zip() compares elements in
    # arbitrary order, which is meaningless. Set INTERSECTION asks the right
    # question: which digits appear in BOTH sets?
    matched_no = ticket_no & user_no
    prize = PAYOUT[len(matched_no)]
    print(f'Ticket digits : {sorted(ticket_no)}')
    print(f'Your digits   : {sorted(user_no)}')
    print(f'Matched {len(matched_no)} digit(s) {sorted(matched_no)} — you win Rs.{prize}')

final_round('0,2,4,6,8,9')

### 2) A tiny banking service
Account state lives in a class; the "menu" is a scripted list of operations. ⚠️ Note the design point in the comments: menus are **loops, not recursion**.

In [ ]:
# Banking Service (deterministic demo)
# The original version drove its menu with input() and hopped between functions
# by calling one another (front -> signup -> front -> ...), used sys.exit() to
# quit (which raises SystemExit in Jupyter), and could loop forever in signup.
# ⚠️ Menus are loops, not recursion — Python has no tail-call optimisation, so
#    every menu hop stacks a new frame until RecursionError (see the
#    "Recursion limits" section above).
import string

PUNC = set(string.punctuation)


def valid_password(pwd):
    """More than 5 characters AND at least one punctuation character."""
    return len(pwd) > 5 and any(ch in PUNC for ch in pwd)


class Account:
    def __init__(self, username, password, balance=0.0):
        self.username = username
        self.password = password
        self.balance = balance

    def deposit(self, amount):
        self.balance += amount
        return f'Deposited Rs.{amount} -> balance Rs.{self.balance}'

    def withdraw(self, amount):
        if amount > self.balance:
            return f'Insufficient balance (Rs.{self.balance})'
        self.balance -= amount
        return f'Withdrew Rs.{amount} -> balance Rs.{self.balance}'

    def check(self):
        return f'Current balance is Rs.{self.balance}'


print('Welcome to Python Bank')
assert valid_password('secret!23'), 'password must be >5 chars with punctuation'
acct = Account('aditya', 'secret!23')

# Scripted list of operations replaces the input()-driven menu:
operations = [('deposit', 500), ('withdraw', 120), ('check', None),
              ('withdraw', 1000), ('deposit', 250), ('check', None)]

for op, amount in operations:
    if op == 'deposit':
        print(acct.deposit(amount))
    elif op == 'withdraw':
        print(acct.withdraw(amount))
    elif op == 'check':
        print(acct.check())

# Interactive variant — note the menu is a while-loop, NOT mutual recursion:
# while True:
#     choice = input('1. Balance  2. Deposit  3. Withdraw  4. Exit: ')
#     if choice == '1':
#         print(acct.check())
#     elif choice == '2':
#         print(acct.deposit(float(input('Amount: '))))
#     elif choice == '3':
#         print(acct.withdraw(float(input('Amount: '))))
#     elif choice == '4':
#         print('Thank you for banking with us')
#         break          # a plain break replaces sys.exit()
#     else:
#         print('Unknown choice')

---

## Common Mistakes & Pitfalls

1. **A mutable default argument** — `def f(x, items=[])`. Defaults are evaluated **once**, at `def` time, so every call shares one list. Use `None` as the sentinel (see **2.7**).
2. **Putting a non-default parameter after a default one** — `def f(a=1, b)` is a `SyntaxError`. Required parameters must come first.
3. **Assuming type hints are enforced.** They are not. `def f(x: int)` happily accepts a string at run time; only a checker like `mypy` will object.
4. **Forgetting `return`.** A function without one returns `None` — so `total = add(2, 3)` silently gives `None` if `add` only *prints*.
5. **Using `global` to share state.** Almost always a sign the value should be a parameter or a return value.
6. **Mutating an argument without saying so.** If a function changes the list it was given, that must be its documented purpose — not a side effect.
7. **Recursion without a reachable base case**, or on data deeper than ~1000 — `RecursionError`.
8. **Naive recursive fibonacci on large `n`.** It is exponential. Add `@cache`.
9. **Shadowing a builtin as a parameter name** — `def f(list, id, type)` breaks them inside the function body.

## Best Practices

- One function, one job. If you need 'and' to describe it, split it.
- Write a docstring for anything that isn't self-evident — say what it *returns*, not how it works.
- Use **keyword-only parameters** (`*`) for options and flags, so calls are self-documenting.
- Use `None` as the default for any parameter whose real default is mutable.
- Annotate public functions with type hints — they are checked documentation.
- Return values rather than printing them; let the caller decide what to do with the result.
- Prefer returning a `NamedTuple` or `dataclass` over a bare tuple once there are 3+ values.
- Use `@cache` / `@lru_cache` for pure functions with repeated inputs.
- Keep functions short enough to see at once — roughly a screenful.

## Practice Exercises

Try these before moving on.

1. Write `area(shape, /, *, width=None, height=None, radius=None)` and explain why `shape` is positional-only and the rest keyword-only.
2. Fix `def append_to(item, target=[]): target.append(item); return target` and prove the fix.
3. Write a function returning min, max and average as a `NamedTuple`; call it and unpack.
4. Add type hints to every function you wrote in **3.x**, then run `mypy` over the file.
5. Compare naive recursive fibonacci against an `@cache` version for `n = 35`. Time both.
6. Write a recursive function to flatten an arbitrarily nested list.
7. Demonstrate all four LEGB levels resolving in one example.
8. Write a function that must not modify its list argument, and prove it doesn't.
9. Rewrite `char_classes` so it also enforces a minimum length and returns `(ok, reasons)` — then decide: tuple, dict, or dataclass?